# Viscosity Prediction Pipeline
Step-by-step analysis of `height_normalized.csv` using `RheologyPipeline`.

## Step 1 — Imports & load raw data

In [ ]:
import sys, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Make sure the repo root is importable
REPO = Path(r"C:\Users\mrast\OneDrive\Documents\GitHub\Automated_Viscometry")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from rheology_pipeline_core import RheologyPipeline, fit_drag_profile, ConeGeometry

# ── Load data ────────────────────────────────────────────────────────────────
CSV_PATH = REPO / "results" / "Auto-runs" / "height_normalized.csv"
df_raw = pd.read_csv(CSV_PATH)

print(f"Loaded: {CSV_PATH.name}")
print(f"Shape : {df_raw.shape}  ({df_raw.shape[0]} height rows, {df_raw.shape[1]-1} silicone samples)")
print()
display(df_raw.head())


In [ ]:
# ── Google style constants ────────────────────────────────────────────────────
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl

# Set Arial font globally
mpl.rcParams['font.family'] = 'Arial'

GOOGLE_COLORS = ["#4285F4", "#EA4335", "#FBBC05", "#34A853", "#FF6D00", "#46BDC6"]

# Continuous colormap: Google Blue → Yellow → Red
GOOGLE_CMAP = LinearSegmentedColormap.from_list(
    "google", [GOOGLE_COLORS[0], GOOGLE_COLORS[2], GOOGLE_COLORS[1]]
)

BORDER = 1.5
FS_LABEL = 20    # axis label font size
FS_TICK  = 14    # tick label font size
FS_LEGEND = 14   # legend font size

def _gs(ax):
    """Apply Google style to an Axes: no grid, thick border, tick size."""
    ax.grid(False)
    ax.tick_params(labelsize=FS_TICK)
    for spine in ax.spines.values():
        spine.set_linewidth(BORDER)

def _cb(mappable, ax, label):
    """Add a styled colorbar."""
    cb = plt.colorbar(mappable, ax=ax)
    cb.set_label(label, fontsize=FS_LEGEND)
    cb.ax.tick_params(labelsize=FS_TICK - 2)
    cb.outline.set_linewidth(BORDER)
    return cb

print("Google style palette & Arial font loaded.")


## Step 1b — Load full_run_260428.csv & plot Drag vs Spindle Height

In [ ]:
import re as _re

# ── Load ──────────────────────────────────────────────────────────────────────
FULL_RUN_PATH = REPO / "results" / "Auto-runs" / "full_run_260428.csv"
df_run = pd.read_csv(
    FULL_RUN_PATH,
    usecols=["Cell_Label", "Z_Height_mm", "RPM", "Rotational_Drag"]
)
df_run["Z_Height_mm"]    = pd.to_numeric(df_run["Z_Height_mm"],    errors="coerce")
df_run["Rotational_Drag"] = pd.to_numeric(df_run["Rotational_Drag"], errors="coerce")
df_run = df_run.dropna(subset=["Z_Height_mm", "Rotational_Drag"])

# Parse nominal viscosity (kcP) from label prefix: "12.5kcP_sdl5" → 12.5
def _kcp(label):
    m = _re.match(r"^([\d.]+)kcP", str(label))
    return float(m.group(1)) if m else np.nan

labels_ordered = sorted(df_run["Cell_Label"].unique(), key=_kcp)
kcp_vals       = [_kcp(l) for l in labels_ordered]

print(f"Loaded : {FULL_RUN_PATH.name}")
print(f"Rows   : {len(df_run):,}")
print(f"Samples: {len(labels_ordered)}")
display(df_run.head(4))

# ── Plot: Drag vs Z-Height, one curve per sample ──────────────────────────────
norm_run = plt.Normalize(vmin=min(kcp_vals), vmax=max(kcp_vals))
sm_run   = plt.cm.ScalarMappable(cmap=GOOGLE_CMAP, norm=norm_run)
sm_run.set_array([])

fig, ax = plt.subplots(figsize=(6, 6))

for label in labels_ordered:
    kcp = _kcp(label)
    sub = df_run[df_run["Cell_Label"] == label].sort_values("Z_Height_mm")
    ax.plot(
        sub["Z_Height_mm"],
        sub["Rotational_Drag"],
        color=GOOGLE_CMAP(norm_run(kcp)),
        lw=2.0,
    )

ax.set_xlabel("Spindle Height — Z-axis  (mm)", fontsize=FS_LABEL)
ax.set_ylabel("Rotational Drag", fontsize=FS_LABEL)
_cb(sm_run, ax, "Nominal viscosity  (kcP)")
_gs(ax)
plt.tight_layout()
plt.show()


## Step 2 — Parse column metadata & visualize raw drag profiles

In [ ]:
RE_COL = re.compile(
    r"^(?P<nom>[\d.]+)kcp_(?P<mu>[\d.]+)_torque_%_rpm_(?P<rpm>[\d.]+)$"
)

samples = []
for col in df_raw.columns:
    if col == "Height":
        continue
    m = RE_COL.match(col)
    if m:
        samples.append({
            "col": col,
            "nom_kcp": float(m["nom"]),
            "mu_cP": float(m["mu"]) * 1000.0,
            "rpm": float(m["rpm"]),
        })

df_meta = pd.DataFrame(samples).sort_values("mu_cP").reset_index(drop=True)
print(f"Parsed {len(df_meta)} silicone samples")
display(df_meta[["col", "nom_kcp", "mu_cP", "rpm"]])

# ── Raw torque vs. height — all samples overlaid ──────────────────────────────
h_raw = df_raw["Height"].values

norm_raw = plt.Normalize(vmin=df_meta["mu_cP"].min(), vmax=df_meta["mu_cP"].max())
sm_raw = plt.cm.ScalarMappable(cmap=GOOGLE_CMAP, norm=norm_raw)
sm_raw.set_array([])

fig, ax = plt.subplots(figsize=(6, 6))
for i, row in df_meta.iterrows():
    torque = df_raw[row["col"]].values
    ax.plot(h_raw, torque, color=GOOGLE_CMAP(norm_raw(row["mu_cP"])), lw=2.0)

ax.set_xlabel("Height  h  (mm)", fontsize=FS_LABEL)
ax.set_ylabel("Torque  T  (%)", fontsize=FS_LABEL)
_cb(sm_raw, ax, "True viscosity  μ  (cP)")
_gs(ax)
plt.tight_layout()
plt.show()


## Step 3 — Fit drag profiles D(h) = A/(h + h_c) + B for each sample

In [ ]:
fit_rows = []

for _, row in df_meta.iterrows():
    h = h_raw - h_raw.min()
    T = df_raw[row["col"]].values
    D = T / row["rpm"]
    fit = fit_drag_profile(h, D, h_c=None)
    fit_rows.append({**row.to_dict(), **fit})

df_fits = pd.DataFrame(fit_rows)
print("Drag-profile fits:")
display(df_fits[["nom_kcp", "mu_cP", "rpm", "A", "h_c", "B", "R2", "n"]].round(4))

# ── Normalised drag curves + fits (all samples overlaid) ─────────────────────
h_plot = np.linspace(0, h_raw.max() - h_raw.min(), 200)
norm_fit = plt.Normalize(vmin=df_fits["mu_cP"].min(), vmax=df_fits["mu_cP"].max())
sm_fit = plt.cm.ScalarMappable(cmap=GOOGLE_CMAP, norm=norm_fit)
sm_fit.set_array([])

fig, ax = plt.subplots(figsize=(6, 6))
for _, row in df_fits.iterrows():
    h = h_raw - h_raw.min()
    D = df_raw[row["col"]].values / row["rpm"]
    D_norm = (D - row["B"]) / row["A"]
    D_fit_norm = 1.0 / (h_plot + row["h_c"])
    color = GOOGLE_CMAP(norm_fit(row["mu_cP"]))
    ax.scatter(h, D_norm, s=10, color=color, alpha=0.45, zorder=2)
    ax.plot(h_plot, D_fit_norm, color=color, lw=2.0, alpha=0.85)

ax.set_xlabel("Height  h  (mm)", fontsize=FS_LABEL)
ax.set_ylabel("Normalised drag  (D − B) / A", fontsize=FS_LABEL)
_cb(sm_fit, ax, "True viscosity  μ  (cP)")
_gs(ax)
plt.tight_layout()
plt.show()


## Step 4 — Load calibration & build A(μ) power-law model

In [ ]:
pipeline = RheologyPipeline()
cal = pipeline.load_silicone_calibration(CSV_PATH)

print("=== Calibration Results ===")
for k, v in cal.items():
    print(f"  {k:25s}: {v:.6g}")

good = df_fits[df_fits["R2"] > 0.7].copy()
mu_range = np.logspace(
    np.log10(good["mu_cP"].min() * 0.8),
    np.log10(good["mu_cP"].max() * 1.2), 200
)
A_fit_line = pipeline.SILICONE_K * mu_range ** pipeline.SILICONE_P

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(good["mu_cP"], good["A"],
           color=GOOGLE_COLORS[0], s=90, edgecolors="k",
           linewidths=0.6, zorder=3, label="Fitted A values")
ax.plot(mu_range, A_fit_line,
        color=GOOGLE_COLORS[1], lw=3, ls="--",
        label=(f"A = {pipeline.SILICONE_K:.2e} · μ^{pipeline.SILICONE_P:.3f}"
               f"  (R²={cal['R2_calibration']:.4f})"))
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("True viscosity  μ  (cP)", fontsize=FS_LABEL)
ax.set_ylabel("Drag amplitude  A", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False)
_gs(ax)
plt.tight_layout()
plt.show()

print(f"\nUniversal h_c = {pipeline.H_C_UNIVERSAL:.4f} mm")
print(f"k = {pipeline.SILICONE_K:.4e},  p = {pipeline.SILICONE_P:.4f}")


## Step 5 — Predict viscosity for every sample via the full pipeline

In [ ]:
pred_rows = []

for _, row in df_meta.iterrows():
    h = h_raw - h_raw.min()
    T = df_raw[row["col"]].values

    result = pipeline.predict_rheology(h, T, row["rpm"])
    pred_rows.append({
        "nom_kcp"      : row["nom_kcp"],
        "true_mu_cP"   : row["mu_cP"],
        "rpm"          : row["rpm"],
        "pred_mu_cP"   : result.get("mu_app_cP", np.nan),
        "regime"       : result.get("regime", "?"),
        "fit_R2"       : result.get("fit_quality", np.nan),
        "A"            : result.get("A", np.nan),
    })

df_pred = pd.DataFrame(pred_rows)
df_pred["error_pct"] = (df_pred["pred_mu_cP"] - df_pred["true_mu_cP"]) / df_pred["true_mu_cP"] * 100

print("Predictions:")
display(df_pred.round(2))


## Step 6 — Accuracy analysis & visualization

In [ ]:
valid = df_pred.dropna(subset=["pred_mu_cP", "true_mu_cP"])

mae      = np.mean(np.abs(valid["error_pct"]))
mape     = mae
rmse_log = np.sqrt(np.mean((np.log10(valid["pred_mu_cP"]) - np.log10(valid["true_mu_cP"])) ** 2))
within10 = (np.abs(valid["error_pct"]) <= 10).mean() * 100
within20 = (np.abs(valid["error_pct"]) <= 20).mean() * 100

from scipy.stats import pearsonr
r, _ = pearsonr(np.log10(valid["true_mu_cP"]), np.log10(valid["pred_mu_cP"]))

print("=" * 45)
print("          ACCURACY REPORT")
print("=" * 45)
print(f"  Samples evaluated : {len(valid)}")
print(f"  MAPE              : {mape:.2f}%")
print(f"  RMSE (log₁₀ cP)  : {rmse_log:.4f}")
print(f"  Pearson r (log)   : {r:.4f}")
print(f"  Within ±10%       : {within10:.1f}%")
print(f"  Within ±20%       : {within20:.1f}%")
print("=" * 45)

# ── Figure 1: Parity plot (log-log) ──────────────────────────────────────────
lim = [valid["true_mu_cP"].min() * 0.7, valid["true_mu_cP"].max() * 1.4]

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(valid["true_mu_cP"], valid["pred_mu_cP"],
                c=valid["error_pct"].abs(), cmap=GOOGLE_CMAP,
                vmin=0, vmax=30, s=100,
                edgecolors="k", linewidths=0.5, zorder=3)
ax.plot(lim, lim, color="k", ls="--", lw=2.5, label="Ideal (1:1)")
ax.fill_between(lim, [v * 0.9 for v in lim], [v * 1.1 for v in lim],
                alpha=0.13, color=GOOGLE_COLORS[3], label="±10%")
ax.fill_between(lim, [v * 0.8 for v in lim], [v * 1.2 for v in lim],
                alpha=0.08, color=GOOGLE_COLORS[0], label="±20%")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("True  μ  (cP)", fontsize=FS_LABEL)
ax.set_ylabel("Predicted  μ  (cP)", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False)
_cb(sc, ax, "|error| %")
_gs(ax)
plt.tight_layout()
plt.show()


In [ ]:
# ── Figure 2: % error vs. true viscosity ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.axhline(0, color="k", lw=1.5)
ax.axhspan(-10, 10, alpha=0.13, color=GOOGLE_COLORS[3], label="±10%")
ax.axhspan(-20, 20, alpha=0.08, color=GOOGLE_COLORS[0], label="±20%")
sc2 = ax.scatter(valid["true_mu_cP"], valid["error_pct"],
                 c=valid["error_pct"].abs(), cmap=GOOGLE_CMAP,
                 vmin=0, vmax=30, s=100,
                 edgecolors="k", linewidths=0.5, zorder=3)
ax.set_xscale("log")
ax.set_xlabel("True  μ  (cP)", fontsize=FS_LABEL)
ax.set_ylabel("Relative error  (%)", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False)
_cb(sc2, ax, "|error| %")
_gs(ax)
plt.tight_layout()
plt.show()


In [ ]:
# ── Figure 3: Error histogram ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.hist(valid["error_pct"], bins=12,
        color=GOOGLE_COLORS[0], edgecolor="white", linewidth=0.8)
ax.axvline(0, color="k", lw=1.5)
ax.axvline(valid["error_pct"].mean(),
           color=GOOGLE_COLORS[1], lw=3, ls="--",
           label=f"mean = {valid['error_pct'].mean():.1f}%")
ax.set_xlabel("Relative error  (%)", fontsize=FS_LABEL)
ax.set_ylabel("Count", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False)
_gs(ax)
plt.tight_layout()
plt.show()


In [ ]:
# ── Figure 4: Drag-profile fit quality (R² per sample) ───────────────────────
bar_colors = [GOOGLE_COLORS[1] if v < 0.95 else GOOGLE_COLORS[0]
              for v in valid["fit_R2"]]

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(len(valid)), valid["fit_R2"].values,
       color=bar_colors, edgecolor="k", linewidth=0.5)
ax.axhline(0.95, color=GOOGLE_COLORS[2], lw=3, ls="--",
           label="R² = 0.95 threshold")
ax.set_xticks(range(len(valid)))
ax.set_xticklabels([f"{v:.0f}" for v in valid["nom_kcp"].values],
                   rotation=90, fontsize=FS_TICK)
ax.set_xlabel("Sample  (kcP nominal)", fontsize=FS_LABEL)
ax.set_ylabel("Drag-profile fit  R²", fontsize=FS_LABEL)
ax.set_ylim(0.98, 1.001)
ax.legend(fontsize=FS_LEGEND, frameon=False)
_gs(ax)
plt.tight_layout()
plt.show()


In [ ]:
# ── Figure 5: True vs. predicted viscosity (grouped bar) ─────────────────────
x = np.arange(len(valid))
w = 0.4

fig, ax = plt.subplots(figsize=(6, 6))
ax.bar(x - w/2, valid["true_mu_cP"].values / 1000, w,
       label="True (kcP)", color=GOOGLE_COLORS[0], edgecolor="k",
       linewidth=0.5, alpha=0.9)
ax.bar(x + w/2, valid["pred_mu_cP"].values / 1000, w,
       label="Predicted (kcP)", color=GOOGLE_COLORS[4], edgecolor="k",
       linewidth=0.5, alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels([f"{v:.0f}" for v in valid["nom_kcp"].values],
                   rotation=90, fontsize=FS_TICK)
ax.set_xlabel("Sample  (nominal kcP)", fontsize=FS_LABEL)
ax.set_ylabel("Viscosity  (kcP)", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False)
_gs(ax)
plt.tight_layout()
plt.show()


In [ ]:

# ── Composite Figure: All subplots in a 4-column grid ──────────────────────────
fig = plt.figure(figsize=(22, 13))
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.40, wspace=0.40)

axes = [
    fig.add_subplot(gs[0, 0]),  # A
    fig.add_subplot(gs[0, 1]),  # B
    fig.add_subplot(gs[0, 2]),  # C
    fig.add_subplot(gs[0, 3]),  # D
    fig.add_subplot(gs[1, 0]),  # E
    fig.add_subplot(gs[1, 1]),  # F
    fig.add_subplot(gs[1, 2]),  # G
]

# ── A: Drag vs Z-Height ──────────────────────────────────────────────────────
ax = axes[0]
for label in labels_ordered:
    kcp = _kcp(label)
    sub = df_run[df_run["Cell_Label"] == label].sort_values("Z_Height_mm")
    ax.plot(sub["Z_Height_mm"], sub["Rotational_Drag"],
            color=GOOGLE_CMAP(norm_run(kcp)), lw=1.8, alpha=0.8)
ax.set_xlabel("Spindle Height Z (mm)", fontsize=FS_LABEL)
ax.set_ylabel("Rotational Drag", fontsize=FS_LABEL)
_gs(ax)
ax.text(-0.08, 1.06, 'A', transform=ax.transAxes, fontsize=18, fontweight='normal', va='top')

# ── B: Raw torque vs. height ─────────────────────────────────────────────────
ax = axes[1]
h_raw = df_raw["Height"].values
for i, row in df_meta.iterrows():
    torque = df_raw[row["col"]].values
    ax.plot(h_raw, torque, color=GOOGLE_CMAP(norm_raw(row["mu_cP"])), lw=1.8, alpha=0.8)
ax.set_xlabel("Height h (mm)", fontsize=FS_LABEL)
ax.set_ylabel("Torque T (%)", fontsize=FS_LABEL)
_gs(ax)
ax.text(-0.08, 1.06, 'B', transform=ax.transAxes, fontsize=18, fontweight='normal', va='top')

# ── C: Normalised drag curves + fits ──────────────────────────────────────────
ax = axes[2]
h_plot = np.linspace(0, h_raw.max() - h_raw.min(), 200)
for _, row in df_fits.iterrows():
    h = h_raw - h_raw.min()
    D = df_raw[row["col"]].values / row["rpm"]
    D_norm = (D - row["B"]) / row["A"]
    D_fit_norm = 1.0 / (h_plot + row["h_c"])
    color = GOOGLE_CMAP(norm_fit(row["mu_cP"]))
    ax.scatter(h, D_norm, s=8, color=color, alpha=0.35, zorder=2)
    ax.plot(h_plot, D_fit_norm, color=color, lw=1.8, alpha=0.8, zorder=2)
ax.set_xlabel("Height h (mm)", fontsize=FS_LABEL)
ax.set_ylabel("Normalised drag (D−B)/A", fontsize=FS_LABEL)
_gs(ax)
ax.text(-0.08, 1.06, 'C', transform=ax.transAxes, fontsize=18, fontweight='normal', va='top')

# ── D: Parity plot (log-log) ─────────────────────────────────────────────────
ax = axes[3]
lim = [valid["true_mu_cP"].min() * 0.7, valid["true_mu_cP"].max() * 1.4]
ax.scatter(valid["true_mu_cP"], valid["pred_mu_cP"],
           c=valid["error_pct"].abs(), cmap=GOOGLE_CMAP,
           vmin=0, vmax=30, s=80, edgecolors="k", linewidths=0.4, zorder=3, alpha=0.8)
ax.plot(lim, lim, color="k", ls="--", lw=2.5, label="Ideal (1:1)")
ax.fill_between(lim, [v * 0.9 for v in lim], [v * 1.1 for v in lim],
                alpha=0.10, color=GOOGLE_COLORS[3], label="±10%")
ax.fill_between(lim, [v * 0.8 for v in lim], [v * 1.2 for v in lim],
                alpha=0.05, color=GOOGLE_COLORS[0], label="±20%")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("True μ (cP)", fontsize=FS_LABEL)
ax.set_ylabel("Predicted μ (cP)", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False, loc='lower right')
_gs(ax)
ax.text(-0.08, 1.06, 'D', transform=ax.transAxes, fontsize=18, fontweight='normal', va='top')

# ── E: % error vs. true viscosity ────────────────────────────────────────────
ax = axes[4]
ax.axhline(0, color="k", lw=1.5, zorder=1)
ax.axhspan(-10, 10, alpha=0.10, color=GOOGLE_COLORS[3], label="±10%")
ax.axhspan(-20, 20, alpha=0.05, color=GOOGLE_COLORS[0], label="±20%")
ax.scatter(valid["true_mu_cP"], valid["error_pct"],
           c=valid["error_pct"].abs(), cmap=GOOGLE_CMAP,
           vmin=0, vmax=30, s=80, edgecolors="k", linewidths=0.4, zorder=3, alpha=0.8)
ax.set_xscale("log")
ax.set_xlabel("True μ (cP)", fontsize=FS_LABEL)
ax.set_ylabel("Relative error (%)", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False, loc='upper right')
_gs(ax)
ax.text(-0.08, 1.06, 'E', transform=ax.transAxes, fontsize=18, fontweight='normal', va='top')

# ── F: Error histogram ───────────────────────────────────────────────────────
ax = axes[5]
ax.hist(valid["error_pct"], bins=12,
        color=GOOGLE_COLORS[0], edgecolor="white", linewidth=0.8, alpha=0.85)
ax.axvline(0, color="k", lw=1.5, zorder=2)
ax.axvline(valid["error_pct"].mean(),
           color=GOOGLE_COLORS[1], lw=3, ls="--", zorder=2,
           label=f"μ = {valid['error_pct'].mean():.1f}%")
ax.set_xlabel("Relative error (%)", fontsize=FS_LABEL)
ax.set_ylabel("Count", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False)
_gs(ax)
ax.text(-0.08, 1.06, 'F', transform=ax.transAxes, fontsize=18, fontweight='normal', va='top')

# ── G: True vs. predicted viscosity (grouped bar) ────────────────────────────
ax = axes[6]
x = np.arange(len(valid))
w = 0.4
ax.bar(x - w/2, valid["true_mu_cP"].values / 1000, w,
       label="True (kcP)", color=GOOGLE_COLORS[0], edgecolor="k",
       linewidth=0.4, alpha=0.85, zorder=2)
ax.bar(x + w/2, valid["pred_mu_cP"].values / 1000, w,
       label="Predicted (kcP)", color=GOOGLE_COLORS[4], edgecolor="k",
       linewidth=0.4, alpha=0.85, zorder=2)
ax.set_xticks(x)
ax.set_xticklabels([f"{v:.0f}" for v in valid["nom_kcp"].values],
                   rotation=90, fontsize=FS_TICK)
ax.set_xlabel("Sample (nominal kcP)", fontsize=FS_LABEL)
ax.set_ylabel("Viscosity (kcP)", fontsize=FS_LABEL)
ax.legend(fontsize=FS_LEGEND, frameon=False)
_gs(ax)
ax.text(-0.08, 1.06, 'G', transform=ax.transAxes, fontsize=18, fontweight='normal', va='top')

# Add main title
fig.suptitle("Viscosity Prediction Pipeline — Complete Analysis", fontsize=24, y=0.998, fontweight='normal')

# Save as SVG
output_svg = REPO / "composite_analysis_figure.svg"
plt.savefig(output_svg, format="svg", bbox_inches="tight", dpi=300)
print(f"Saved: {output_svg.name}")

plt.show()
